In [1]:
import os
import re
import json
import glob
import itertools
import h5py
import numpy as np
import matplotlib.pyplot as plt
from opt_einsum import contract


hadron = "pion"          #"pion" or "proton"
frame = "forward"        #"forward" or "sym"
q = (0,0,0)              #forward: (0,0,0);  sym: (2,0,0), (-2,0,0), (0,2,0) or (0,-2,0)
project = True           #proton only: True = positive-parity files, False = open Dirac files (none on this laptop)


pt2_dir = "/Users/mcp3270/Desktop/GLUON_ANALYSIS_MANUAL/local_analysis/final_production_delta"
ncfg = 250
#C3input: q-phase averaged in;  C2pf, C2pi: no q-phase averaged in;  at q = 0 the C3input file is also the C2pf and the C2pi file
pt2_template = os.path.join(pt2_dir, "2pt_coulomb_rho4p5z2p25-rho1p5_frac0p8-0_smear_GEVP_{frame}_q{qx}_{qy}_{qz}_{pt2_type}_fb_tsep20_3072src_ncfg{ncfg}_{hadron}.h5")

assert frame in ["forward", "sym"], f"frame = {frame}: use forward or sym"
assert (frame == "forward") == (tuple(q) == (0,0,0)), f"frame = {frame} with q = {q}: forward needs q = (0,0,0), sym a nonzero q"

if hadron == "pion":
    hadron_tag = "pion"
elif project:
    hadron_tag = "proton_projected"
else:
    hadron_tag = "proton"


#functions
def jk(data, jk_axis=0):
    n = data.shape[jk_axis]
    data = np.moveaxis(data, jk_axis, 0)
    mean = np.mean(data, axis=0)
    err = np.sqrt((n - 1.0) / n * np.sum((data - mean) ** 2, axis=0))
    return mean, err


def jackknife(x):
    n_cfg = x.shape[0]
    return (x.sum(axis=0) - x) / (n_cfg - 1)


def read_pt2(pt2_type):
    pt2_file = pt2_template.format(frame=frame, qx=q[0], qy=q[1], qz=q[2], pt2_type=pt2_type, ncfg=ncfg, hadron=hadron_tag)
    print(pt2_file)
    assert os.path.exists(pt2_file), f"no such file: {pt2_file}"
    with h5py.File(pt2_file, "r") as f:
        for name in f:
            print(name, f[name].shape, f[name].dtype)
        return f["pt2_forward"][()], f["pt2_backward"][()]


pt2_forward_C3input, pt2_backward_C3input = read_pt2("C3input")
if frame == "forward":
    pt2_forward_C2pf, pt2_backward_C2pf = pt2_forward_C3input, pt2_backward_C3input
    pt2_forward_C2pi, pt2_backward_C2pi = pt2_forward_C3input, pt2_backward_C3input
if frame == "sym":
    pt2_forward_C2pf, pt2_backward_C2pf = read_pt2("C2pf")
    pt2_forward_C2pi, pt2_backward_C2pi = read_pt2("C2pi")

/Users/mcp3270/Desktop/GLUON_ANALYSIS_MANUAL/local_analysis/final_production_delta/2pt_coulomb_rho4p5z2p25-rho1p5_frac0p8-0_smear_GEVP_forward_q0_0_0_C3input_fb_tsep20_3072src_ncfg250_pion.h5
cfg_list (250,) int64
momentum_list (7, 3) int64
pf_list (7, 3) int64
pi_list (7, 3) int64
pt2_backward (250, 2, 2, 3, 3, 24, 7, 20) complex128
pt2_forward (250, 2, 2, 3, 3, 24, 7, 20) complex128
pz_list (7,) int64
q (3,) int64
smear_list (2, 3) float64
smear_names (2,) object
source_layout (250,) object
t_src_list (250, 24) int64
x_src_list (250, 4) int64
y_src_list (250, 4) int64
z_src_list (250, 8) int64


In [2]:
#======== GEVP ========
t0 = 3                 #GEVP reference time
t_ref = 5              #time of the ground-state vector v0
c_sign = {"pion": -1, "proton": +1}[hadron]     #the pion comes out negative, the projected proton positive
#the GEVP basis: (smear, gamma);  smear 0 = B [4.5,2.25,0.8], 1 = U [1.5,1.5,0];  gamma 0 = G5, 1 = G5G4, 2 = G3G5
op_basis = [(0,0),(0,1),(0,2),(1,0),(1,1),(1,2)]
assert pt2_forward_C2pf.ndim == 8, "the GEVP needs (cfg, smear, smear, gamma, gamma, t_src, pz, tsep): open Dirac files have to be projected first"


def op_matrix(pt2):
    #(cfg, smear_snk, smear_src, gamma_snk, gamma_src, ...) -> (cfg, op_snk, op_src, ...)
    n_op = len(op_basis)
    Cij = np.zeros(pt2.shape[:1] + (n_op, n_op) + pt2.shape[5:], dtype=complex)
    for a, (s_snk, g_snk) in enumerate(op_basis):
        for b, (s_src, g_src) in enumerate(op_basis):
            Cij[:, a, b] = pt2[:, s_snk, s_src, g_snk, g_src]
    return Cij


def solve_gevp(C_matrix):
    #C_matrix (op, op, pz, tsep): eigenvalues (pz, state, tsep) and eigenvectors (pz, state, tsep, op), largest eigenvalue first, NaN at tsep <= t0
    n_op, _, n_pz, n_tsep = C_matrix.shape
    eigen_val = np.full((n_pz, n_op, n_tsep), np.nan)
    eigen_vec = np.full((n_pz, n_op, n_tsep, n_op), np.nan, dtype=complex)
    for ipz in range(n_pz):
        L_inv = np.linalg.inv(np.linalg.cholesky(C_matrix[:, :, ipz, t0]))
        for t in range(t0 + 1, n_tsep):
            M = L_inv @ C_matrix[:, :, ipz, t] @ L_inv.conj().T
            w, u = np.linalg.eigh(M)
            v = L_inv.conj().T @ u
            order = np.argsort(w)[::-1]
            eigen_val[ipz, :, t] = w[order]
            eigen_vec[ipz, :, t] = v[:, order].T
    return eigen_val, eigen_vec


def apply_v0(Cij, v):
    #Cij (cfg, op, op, ..., pz, tsep), v (pz, op) -> (cfg, ..., pz, tsep)
    return c_sign * np.einsum("pa,cab...pT,pb->c...pT", v.conj(), Cij, v)


pt2_input = {"forward":  {"C3input": pt2_forward_C3input,  "C2pf": pt2_forward_C2pf,  "C2pi": pt2_forward_C2pi},
             "backward": {"C3input": pt2_backward_C3input, "C2pf": pt2_backward_C2pf, "C2pi": pt2_backward_C2pi}}
eigen_val, v0, pt2_rec = {}, {}, {}
for half in pt2_input:
    #C2pf and C2pi, average over t_src: (cfg, op, op, pz, tsep)
    C2pf = op_matrix(pt2_input[half]["C2pf"].mean(axis=5))
    C2pi = op_matrix(pt2_input[half]["C2pi"].mean(axis=5))
    #average over cfg of (C2pf + C2pi)/2, positive sign, hermitian part: (op, op, pz, tsep)
    C_mean = c_sign * 0.5 * (C2pf + C2pi).mean(axis=0)
    C_mean = 0.5 * (C_mean + C_mean.conj().swapaxes(0, 1))
    eigen_val[half], eigen_vec = solve_gevp(C_mean)
    v0[half] = eigen_vec[:, 0, t_ref, :]            #(pz, op), the same vector at sink and source, so its phase cancels
    #reconstructed 2pt: C2pf, C2pi (cfg, pz, tsep);  C3input keeps t_src (cfg, t_src, pz, tsep)
    pt2_rec[half] = {"C2pf": apply_v0(C2pf, v0[half]),
                     "C2pi": apply_v0(C2pi, v0[half]),
                     "C3input": apply_v0(op_matrix(pt2_input[half]["C3input"]), v0[half])}

#check: effective mass of the reconstructed C2pf, forward and backward averaged: (pz, tsep - 1)
c2_jk = jackknife(0.5 * (pt2_rec["forward"]["C2pf"] + pt2_rec["backward"]["C2pf"]).real)
meff_mean, meff_err = jk(np.log(c2_jk[:, :, :-1] / c2_jk[:, :, 1:]))
print("meff at t = 8, pz = 0 .. 6:")
print(" ".join(f"{m:.4f}({e * 1e4:.0f})" for m, e in zip(meff_mean[:, 8], meff_err[:, 8])))



meff at t = 8, pz = 0 .. 6:
0.1411(3) 0.2420(4) 0.4144(14) 0.5976(37) 0.7832(73) 0.9391(165) 0.9933(659)


/var/folders/fd/b0prbcvx72l2qq4ncklr4nq40000gn/T/ipykernel_84454/3171239388.py:61: RuntimeWarning: invalid value encountered in log
  meff_mean, meff_err = jk(np.log(c2_jk[:, :, :-1] / c2_jk[:, :, 1:]))


In [3]:
#======== 3pt, method 1 ========
hyp = 8
Lt = 96
tau_list = np.arange(20)
half_sign = {"forward": +1, "backward": -1}     #operator at t_src + tau (forward), t_src - tau (backward)

C3input_file = pt2_template.format(frame=frame, qx=q[0], qy=q[1], qz=q[2], pt2_type="C3input", ncfg=ncfg, hadron=hadron_tag)
with h5py.File(C3input_file, "r") as f:
    cfg_list = f["cfg_list"][()]
    t_src_list = f["t_src_list"][()]            #(cfg, t_src): the slice of each t_src index, not sorted, shifts per cfg

#FF (cfg, munu, rhosig, hyp, w, t), munu order XY, YZ, XZ, XT, YT, ZT
FF_file = os.path.join(pt2_dir, f"FF_hyp0-8_w0-9_q{q[0]}_{q[1]}_{q[2]}_ncfg800.h5")
with h5py.File(FF_file, "r") as f:
    assert tuple(f["q"][()]) == tuple(q)
    FF_cfg = f["cfg_list"][()]
    idx = np.searchsorted(FF_cfg, cfg_list)     #the 2pt cfgs are not the first 250 FF cfgs
    assert np.array_equal(FF_cfg[idx], cfg_list)
    ihyp = list(f["hyp_list"][()]).index(hyp)
    w_list = f["w_list"][()]
    TXTX = f["FF"][idx, 3, 3, ihyp]
    TYTY = f["FF"][idx, 4, 4, ihyp]
    XYXY = f["FF"][idx, 0, 0, ihyp]
O = TXTX + TYTY - 2 * XYXY                      #(cfg, w, t)

n_cfg, n_tsrc = t_src_list.shape
i_cfg = np.arange(n_cfg)[:, None, None]
pt3_jk = {}
for half in half_sign:
    C2 = pt2_rec[half]["C3input"]                                       #(cfg, t_src, pz, tsep), c_sign already in
    t_op = (t_src_list[:, :, None] + half_sign[half] * tau_list) % Lt   #(cfg, t_src, tau)
    O_tau = O[i_cfg, :, t_op]                                           #(cfg, t_src, tau, w): numpy puts w last here
    connected = np.einsum("cspT,cstw->cwpTt", C2, O_tau, optimize=True) / n_tsrc    #(cfg, w, pz, tsep, tau)
    pt2_avg = C2.mean(axis=1)                                           #(cfg, pz, tsep)
    op_avg = O_tau.mean(axis=1)                                         #(cfg, tau, w)
    pt3_jk[half] = jackknife(connected) - np.einsum("cpT,ctw->cwpTt", jackknife(pt2_avg), jackknife(op_avg), optimize=True)    #(jk, w, pz, tsep, tau)

#check: C3/C2 at tsep 8, tau 4, w 0
for half in half_sign:
    m, e = jk(pt3_jk[half][:, 0, :, 8, 4].real / jackknife(pt2_rec[half]["C2pf"]).real[:, :, 8])
    print(half, " ".join(f"{a:+.3f}({b * 1e3:.0f})" for a, b in zip(m, e)))

forward -0.145(22) -0.279(28) -0.468(53) -0.546(95) -0.486(142) -0.462(216) -0.645(844)
backward -0.159(22) -0.273(28) -0.432(49) -0.444(84) -0.316(135) -0.315(224) +0.512(918)


In [4]:
#======== ratio ========
#R(tsep, tau) = C3(tsep, tau) / C2pf(tsep) * sqrt[ C2pi(tsep - tau) C2pf(tau) C2pf(tsep) / (C2pf(tsep - tau) C2pi(tau) C2pi(tsep)) ]
#the ground state gives M / (2 sqrt(E_f E_i)) at every tsep, tau;  at q = 0 the square root is 1
n_tsep = pt3_jk["forward"].shape[3]
tsep_grid, tau_grid = np.meshgrid(np.arange(n_tsep), tau_list, indexing="ij")   #(tsep, tau)
in_window = tau_grid <= tsep_grid
rest_grid = np.where(in_window, tsep_grid - tau_grid, 0)                        #tsep - tau, the 0 is only used where R is set to NaN


def build_ratio(pt3, c2f, c2i):
    #pt3 (jk, w, pz, tsep, tau), c2f and c2i real (jk, pz, tsep) -> R (jk, w, pz, tsep, tau), complex, NaN for tau > tsep
    root = np.sqrt(c2i[:, :, rest_grid] * c2f[:, :, tau_grid] * c2f[:, :, tsep_grid]
                   / (c2f[:, :, rest_grid] * c2i[:, :, tau_grid] * c2i[:, :, tsep_grid]))    #(jk, pz, tsep, tau)
    return np.where(in_window, pt3 * (root / c2f[:, :, tsep_grid])[:, None], np.nan)


c2f_jk = {half: jackknife(pt2_rec[half]["C2pf"]).real for half in half_sign}    #(jk, pz, tsep)
c2i_jk = {half: jackknife(pt2_rec[half]["C2pi"]).real for half in half_sign}
ratio_jk = {half: build_ratio(pt3_jk[half], c2f_jk[half], c2i_jk[half]) for half in half_sign}
#forward and backward together: average the jackknife 3pt and 2pt first, then the ratio
ratio_jk["fb"] = build_ratio(0.5 * (pt3_jk["forward"] + pt3_jk["backward"]),
                             0.5 * (c2f_jk["forward"] + c2f_jk["backward"]),
                             0.5 * (c2i_jk["forward"] + c2i_jk["backward"]))

#check: Re R at tsep 8, tau 4, w 0
for half in ratio_jk:
    m, e = jk(ratio_jk[half][:, 0, :, 8, 4].real)
    print(f"{half:8s}", " ".join(f"{a:+.3f}({b * 1e3:.0f})" for a, b in zip(m, e)))

forward  -0.145(22) -0.279(28) -0.468(53) -0.546(95) -0.486(142) -0.462(216) -0.645(844)
backward -0.159(22) -0.273(28) -0.432(49) -0.444(84) -0.316(135) -0.315(224) +0.512(918)
fb       -0.152(21) -0.276(27) -0.450(49) -0.495(84) -0.401(128) -0.388(179) -0.084(658)
